### NLP Pipeline for E2E Text Classification

#### 0- Reading Data

#### About Dataset
IMDB dataset having 50K movie reviews for natural language processing or Text analytics
Link: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/data

In [14]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [29]:
df = pd.read_csv('IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


#### 1- Text Preprocessing

In [ ]:
import pandas as pd
import html
import unicodedata
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download necessary resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))

# all preprocessing needed functions
def remove_special_chars(text):
    re1 = re.compile(r'  +')
    x1 = text.lower().replace('#39;', "'").replace('amp;', '&').replace('#146;', "'").replace(
        'nbsp;', ' ').replace('#36;', '$').replace('\\n', "\n").replace('quot;', "'").replace(
        '<br />', "\n").replace('\\"', '"').replace('<unk>', 'u_n').replace(' @.@ ', '.').replace(
        ' @-@ ', '-').replace('\\', ' \\ ')
    return re1.sub(' ', html.unescape(x1))

def remove_non_ascii(text):
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')

def to_lowercase(text):
    return text.lower()

def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

def replace_numbers(text):
    return re.sub(r'\d+', '', text)

def text2words(text):
    return word_tokenize(text)

def remove_stopwords(words):
    return [word for word in words if word not in stop_words]

def lemmatize_words(words):
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(word) for word in words]

def normalize_text(text):
    """Applies all preprocessing steps to text"""
    text = remove_special_chars(text)
    text = remove_non_ascii(text)
    text = remove_punctuation(text)
    text = to_lowercase(text)
    #text = replace_numbers(text)
    words = text2words(text)
    words = remove_stopwords(words)
    words = lemmatize_words(words)

    return ' '.join(words)  # Converting list of words back to string

# Apply the complete text preprocessing pipeline
df['cleaned_review'] = df['review'].apply(normalize_text)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Ahmed.Nabawi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ahmed.Nabawi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Ahmed.Nabawi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [31]:
# Display the cleaned reviews
df[['review', 'cleaned_review']].head(10)

,review,cleaned_review
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...
5,"Probably my all-time favorite movie, a story o...",probably alltime favorite movie story selfless...
6,I sure would like to see a resurrection of a u...,sure would like see resurrection dated seahunt...
7,"This show was an amazing, fresh & innovative i...",show amazing fresh innovative idea first aired...
8,Encouraged by the positive comments about this...,encouraged positive comment film looking forwa...
9,If you like original gut wrenching laughter yo...,like original gut wrenching laughter like movi...


#### 2- Feature Extraction and Vectorization

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=10000)  # Limit the number of features to reduce memory usage
X = tfidf.fit_transform(df['cleaned_review'])  # Convert text to TF-IDF vectors

print(X.shape)  # Check the shape of the TF-IDF matrix

# Convert sparse matrix to a dense format for readability (only first 5 rows)
df_tfidf = pd.DataFrame(X[:5].toarray(), columns=tfidf.get_feature_names_out())

# Display the first 5 rows in DataFrame format
print(df_tfidf)

(50000, 10000)
   aaron  abandon  abandoned  abbey  abbott  abc  abducted  ability  able  \
0    0.0      0.0        0.0    0.0     0.0  0.0       0.0      0.0   0.0   
1    0.0      0.0        0.0    0.0     0.0  0.0       0.0      0.0   0.0   
2    0.0      0.0        0.0    0.0     0.0  0.0       0.0      0.0   0.0   
3    0.0      0.0        0.0    0.0     0.0  0.0       0.0      0.0   0.0   
4    0.0      0.0        0.0    0.0     0.0  0.0       0.0      0.0   0.0   

   ably  ...  zhang  zizek  zoey  zombi    zombie  zone  zoo  zoom  zorro  \
0   0.0  ...    0.0    0.0   0.0    0.0  0.000000   0.0  0.0   0.0    0.0   
1   0.0  ...    0.0    0.0   0.0    0.0  0.000000   0.0  0.0   0.0    0.0   
2   0.0  ...    0.0    0.0   0.0    0.0  0.000000   0.0  0.0   0.0    0.0   
3   0.0  ...    0.0    0.0   0.0    0.0  0.212362   0.0  0.0   0.0    0.0   
4   0.0  ...    0.0    0.0   0.0    0.0  0.000000   0.0  0.0   0.0    0.0   

   zucco  
0    0.0  
1    0.0  
2    0.0  
3    0.0  
4   

#### 3- Model Selection

###### a. Splitting Data into Train & Test Sets

In [ ]:
from sklearn.model_selection import train_test_split

df_one_hot = pd.get_dummies(df, columns=['sentiment'], dtype=int)
y = df_one_hot['sentiment_positive'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(40000, 10000) (10000, 10000) (40000,) (10000,)


###### b. Model

In [35]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


LogisticRegression(max_iter=1000)

#### 4- Model Prediction

In [37]:
y_pred = model.predict(X_test)
y_pred

array([0, 1, 0, ..., 1, 0, 1])

#### 5- Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Evaluate model performance
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')

# Detailed classification report
print(classification_report(y_test, y_pred))

Accuracy: 0.8927
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      4961
           1       0.88      0.91      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



#### --> trying several models to see which one is the best

In [ ]:
#### 